# HydroSense-Kenya - Level 6: AI-Assisted Programming, Testing & Reproducibility
**Course:** ICS 2207 Scientific Computing  
**Level:** 6 of 6 (15 marks)

This level demonstrates:
1. Transparent, validated use of AI tools
2. Automated testing with pytest
3. Reproducibility - anyone can clone and run this project
4. Scientific summary and reflection

---

## 1. AI Use - What We Used, How, and How We Verified It

This project used **Claude (Anthropic)** as an AI programming assistant at several stages.
All uses are documented in `AI_USE_LOG.md` at the project root. The key uses are summarised below.

### AI Use Instance 1 - Generating pytest test cases

**Prompt given to AI:**  
*"Write pytest tests for a bisection root-finding function that takes f, a, b, tol, max_iter and returns root, iters, history. Cover: correct root for quadratic, raises ValueError when f(a)*f(b)>0, error decreases over iterations, output length is 3."*

**What AI generated:**  
A test class with 6 test methods covering the specified cases.

**How we modified it:**  
Added tests specific to the irrigation deficit function (our actual domain problem), and added the `TestRootConsistency` class to cross-verify all three methods agree - this was not in the AI output.

**How we validated it:**  
Ran `pytest tests/test_numerical_methods.py -v` and verified all tests passed. Then deliberately broke the bisection function (removed the sign check) to confirm the `test_raises_on_same_sign` test correctly fails.

---

### AI Use Instance 2 - Writing docstrings for numerical methods

**Prompt given to AI:**  
*"Write a numpy-style docstring for a function called gaussian_elimination(A, b) that solves Ax=b using Gaussian elimination with partial pivoting. Include Parameters, Returns, Raises, and a short example."*

**What AI generated:**  
A complete docstring with Parameters, Returns, Raises, Notes, and Example sections.

**How we modified it:**  
Added the specific note about partial pivoting improving numerical stability (the AI mentioned pivoting but did not explain why it matters for floating point). We also added the formula description and the irrigation-specific Notes section.

**How we validated it:**  
Read the docstring against the actual function implementation line by line and confirmed every parameter description matches the code. Ran `help(gaussian_elimination)` in a Python session to verify it renders correctly.

---

### What AI Was NOT Used For

- The mathematical derivations (water balance equation, ET formula, convergence analysis)
- The numerical method implementations themselves - bisection, Newton-Raphson, secant, Gaussian elimination, Euler, RK4 were all coded manually
- The scientific interpretations of every plot
- The Monte Carlo and optimisation logic
- The problem statement and data dictionary

**Reflection:** AI tools are useful for boilerplate (docstrings, test scaffolding) but cannot substitute for understanding the mathematics or making scientific judgements. Every AI output required careful reading and modification before it was usable.

## 2. Running the Full Test Suite

In [ ]:
import subprocess
import sys

# Run pytest from the project root and capture output
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True,
    text=True,
    cwd='..'
)

print(result.stdout)
if result.returncode != 0:
    print('FAILURES:')
    print(result.stderr)

## 3. Test Coverage Overview

| Test File | What It Tests | Test Count |
|-----------|--------------|------------|
| `test_numerical_methods.py` | Bisection, Newton-Raphson, Secant, Finite differences, Trapezoidal, Simpson, Gaussian elimination | 45+ tests |
| `test_water_balance.py` | ET formula, water balance physics, stress detection, irrigation needed | 20+ tests |
| `test_simulation.py` | Euler, RK4, Monte Carlo, Optimisation | 30+ tests |
| `test_data_cleaning.py` | Raw data audit, cleaned weather, cleaned soil | 25+ tests |

**Total: 120+ automated tests across the full pipeline.**

### Testing Strategy

We follow three testing principles:

1. **Correctness against known values** - for example, the trapezoidal integral of `f(x)=x` from 0 to 4 must equal exactly 8.0. We compute this analytically and assert our implementation matches.

2. **Physical validity** - soil moisture can never be negative or exceed field capacity. These are hard physical constraints that every function must respect, tested with random inputs.

3. **Consistency across methods** - bisection, Newton-Raphson, and secant must all agree on the same root to within tolerance. If they don't, at least one implementation is wrong.

## 4. Reproducing All Results from Scratch

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from src.numerical_methods import bisection, newton_raphson, trapezoidal, simpson, gaussian_elimination
from src.simulation import euler_simulate, rk4_simulate, monte_carlo_simulate, optimise_irrigation

os.makedirs('../outputs', exist_ok=True)

# ── Step 1: Load official datasets ──
weather = pd.read_csv('../data/raw/weather_daily.csv', na_values=['NA', ''])
soil    = pd.read_csv('../data/raw/soil_sensor_data.csv', na_values=['NA', ''])
params  = pd.read_csv('../data/raw/crop_zone_parameters.csv')

# ── Step 2: Clean weather ──
weather['rainfall_mm']   = weather['rainfall_mm'].fillna(0.0)
rolling                  = weather['temperature_c'].rolling(7, center=True, min_periods=3).mean()
outlier_idx              = weather[weather['temperature_c'] > 40].index
weather.loc[outlier_idx, 'temperature_c'] = rolling[outlier_idx].values
weather['humidity_pct']  = weather['humidity_pct'].fillna(weather['humidity_pct'].mean())

# ── Step 3: Clean soil ──
fault_mask = (soil['zone_id'] == 'Zone_B') & (soil['soil_moisture_pct'] < 10)
soil.loc[fault_mask, 'soil_moisture_pct'] = np.nan
for zone in ['Zone_A','Zone_B','Zone_C']:
    mask = soil['zone_id'] == zone
    soil.loc[mask, 'soil_moisture_pct'] = soil.loc[mask, 'soil_moisture_pct'].interpolate()
check_mask = soil['sensor_status'] == 'CHECK'
soil.loc[check_mask, 'pump_flow_lpm'] = np.nan
tank_fault = (soil['zone_id'] == 'Zone_C') & (soil['tank_level_liters'] > 9000)
soil.loc[tank_fault, 'tank_level_liters'] = int(soil[~tank_fault & (soil['zone_id']=='Zone_C')]['tank_level_liters'].mean())

# ── Step 4: Compute ET ──
T     = weather['temperature_c'].values
W     = weather['wind_speed_mps'].values
S     = weather['solar_index'].values
H     = weather['humidity_pct'].values
Rain  = weather['rainfall_mm'].values
ET    = np.maximum(0.0, 0.12*T + 0.35*W + 2.4*S - 0.025*H)

print('Pipeline ready. Key statistics:')
print(f'  Mean daily ET:        {ET.mean():.3f} mm/day')
print(f'  Total March rainfall: {Rain.sum():.1f} mm')
print(f'  Total ET:             {ET.sum():.1f} mm')
print(f'  Net water balance:    {Rain.sum()-ET.sum():.1f} mm (negative = deficit)')

In [ ]:
# ── Step 5: Key Results - Root Finding ──
zone_a_params = params[params['zone_id']=='Zone_A'].iloc[0]
S_t    = soil[soil['zone_id']=='Zone_A']['soil_moisture_pct'].iloc[0]
R_t    = Rain[0]
ET_t   = ET[0]

def irr_f(I, _S=S_t, _R=R_t, _ET=ET_t,
           _dc=zone_a_params['drainage_coefficient'],
           _fc=zone_a_params['field_capacity_pct'],
           _tgt=zone_a_params['target_moisture_pct']):
    drain  = _dc * max(0, _S - _fc)
    return _S + _R + I - _ET - drain - _tgt

root_b, iters_b, _ = bisection(irr_f, 0, 50)
root_n, iters_n, _ = newton_raphson(irr_f, lambda I: 1.0, x0=10.0)

print('ROOT FINDING - Zone A, Day 1:')
print(f'  Bisection:      I = {root_b:.4f} mm  ({iters_b} iterations)')
print(f'  Newton-Raphson: I = {root_n:.4f} mm  ({iters_n} iterations)')
print(f'  Methods agree:  {abs(root_b - root_n) < 0.001}')

In [ ]:
# ── Step 6: Simulation ──
zone_params = {row['zone_id']: row for _, row in params.iterrows()}
irr_zero    = np.zeros(30)

print('SIMULATION RESULTS (30-day, no irrigation):')
for zone in ['Zone_A','Zone_B','Zone_C']:
    zp  = zone_params[zone]
    S0  = soil[soil['zone_id']==zone]['soil_moisture_pct'].iloc[0]
    rk4 = rk4_simulate(S0, Rain, ET, irr_zero, zp['drainage_coefficient'], zp['field_capacity_pct'])
    stress = (rk4[1:] < zp['min_moisture_pct']).sum()
    print(f'  {zone} ({zp["crop_type"]}): S_final={rk4[-1]:.2f}%  stress_days={stress}')

In [ ]:
# ── Step 7: Optimised schedules ──
print('OPTIMISATION RESULTS:')
for zone in ['Zone_A','Zone_B','Zone_C']:
    zp  = zone_params[zone]
    S0  = soil[soil['zone_id']==zone]['soil_moisture_pct'].iloc[0]
    opt = optimise_irrigation(
        S0, Rain, ET,
        zp['drainage_coefficient'], zp['field_capacity_pct'],
        zp['min_moisture_pct'],     zp['target_moisture_pct']
    )
    print(f'  {zone}: total_irrigation={opt["total_water"]:.1f}mm  stress_days={opt["stress_days"]}  water_saved={opt["savings_vs_max"]:.1f}mm')

In [ ]:
# ── Step 8: Monte Carlo summary ──
print('MONTE CARLO RESULTS (1000 scenarios, 30% rainfall noise):')
for zone in ['Zone_A','Zone_B','Zone_C']:
    zp  = zone_params[zone]
    S0  = soil[soil['zone_id']==zone]['soil_moisture_pct'].iloc[0]
    mc  = monte_carlo_simulate(
        S0, Rain, ET, irr_zero,
        zp['drainage_coefficient'], zp['field_capacity_pct'], zp['min_moisture_pct'],
        n_scenarios=1000, noise_std=0.3, seed=42
    )
    print(f'  {zone}: shortage_prob={mc["shortage_prob"]*100:.1f}%  '
          f'final_mean={mc["mean"][-1]:.1f}%  '
          f'final_p5={mc["p5"][-1]:.1f}%')

## 5. Final Scientific Summary - All 6 Levels

### The Problem
A demonstration farm in Kenya needs a data-driven irrigation system. Three crop zones (tomato, kale, maize) are monitored by weather stations and soil sensors. The goal: use 30 days of data to model soil water availability, quantify uncertainty, and produce an efficient irrigation schedule.

### Methods and Key Results

| Level | Method | Key Finding |
|-------|--------|-------------|
| 1 | ET formula + water balance | 30th March 2026 ET (≈130mm) significantly exceeds rainfall on 23/30 days |
| 2 | Vectorization + error propagation | NumPy is ~30× faster than loops; 10% sensor noise shifts ET by 0.3mm/day |
| 3 | Root finding, integration, linear systems | Newton-Raphson finds irrigation target in 2 iterations; Zone C has largest cumulative deficit |
| 4 | Data cleaning + 5 visualisations | 7 data quality issues found and corrected with documented justifications |
| 5 | Euler vs RK4, Monte Carlo, optimisation | Zone C has >60% stress probability without irrigation; optimised schedule saves 30–45% water |
| 6 | Testing + reproducibility | 120+ automated tests; full pipeline reproducible from `pytest tests/ -v` |

### Limitations and Future Work

1. **ET formula** - the simplified empirical formula underestimates ET on high-stress days compared to the full Penman-Monteith equation. Future work should implement the FAO standard.

2. **Optimisation** - the greedy daily strategy has no lookahead. A multi-day planning horizon (e.g., model predictive control using weather forecasts) would improve water efficiency further.

3. **Data length** - 30 days is insufficient for seasonal analysis. Extending to a full year would allow fitting crop stage-specific ET multipliers.

4. **Monte Carlo noise model** - we use log-normal multiplicative noise. A location-specific rainfall distribution (e.g., gamma distribution fitted to Kenyan historical rainfall) would be more realistic.

### Recommendations for the Farm

- **Zone C (maize)** requires priority attention - highest stress risk and largest area. Install a secondary tank or increase pump allocation.
- **Irrigate in the final week of March** regardless of rainfall forecast - this is when all zones are most depleted and the tank still has reserve capacity.
- **Calibrate temperature sensors annually** - the 45.8°C outlier on day 14 shows that sensor drift can introduce significant ET estimation errors across the season.
- **Adopt the optimised schedule** - it saves approximately 35% of water vs always-on irrigation while keeping all zones above stress threshold in the mean scenario.

In [ ]:
# ── Final summary plot - all key results in one figure ──
colors = {'Zone_A': '#E74C3C', 'Zone_B': '#2ECC71', 'Zone_C': '#3498DB'}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# ── Top row: optimised moisture trajectories per zone ──
for idx, zone in enumerate(['Zone_A','Zone_B','Zone_C']):
    zp  = zone_params[zone]
    S0  = soil[soil['zone_id']==zone]['soil_moisture_pct'].iloc[0]
    rk4 = rk4_simulate(S0, Rain, ET, irr_zero, zp['drainage_coefficient'], zp['field_capacity_pct'])
    opt = optimise_irrigation(S0, Rain, ET, zp['drainage_coefficient'], zp['field_capacity_pct'],
                              zp['min_moisture_pct'], zp['target_moisture_pct'])
    days = range(31)
    ax   = axes[0, idx]
    ax.plot(days, rk4, 'r--', linewidth=1.5, alpha=0.7, label='No irrigation')
    ax.plot(days, opt['moisture'], color=colors[zone], linewidth=2.5, label='Optimised')
    ax.axhline(zp['min_moisture_pct'],    color='orange', linestyle=':', linewidth=1)
    ax.axhline(zp['target_moisture_pct'], color='green',  linestyle='--', linewidth=1)
    ax.set_title(f'{zone} - {zp["crop_type"]}')
    ax.set_ylabel('Moisture (%)')
    ax.set_xlabel('Day')
    ax.legend(fontsize=8)

# ── Bottom left: Water savings comparison ──
zones_list = ['Zone_A','Zone_B','Zone_C']
opt_water  = []
max_water  = []
for zone in zones_list:
    zp  = zone_params[zone]
    S0  = soil[soil['zone_id']==zone]['soil_moisture_pct'].iloc[0]
    opt = optimise_irrigation(S0, Rain, ET, zp['drainage_coefficient'], zp['field_capacity_pct'],
                              zp['min_moisture_pct'], zp['target_moisture_pct'])
    opt_water.append(opt['total_water'])
    max_water.append(opt['total_water'] + opt['savings_vs_max'])

x = np.arange(3)
w = 0.35
axes[1, 0].bar(x - w/2, max_water, w, alpha=0.5, color=[colors[z] for z in zones_list], label='Always-on')
axes[1, 0].bar(x + w/2, opt_water, w, alpha=0.9, color=[colors[z] for z in zones_list], label='Optimised')
axes[1, 0].set_xticks(x); axes[1, 0].set_xticklabels(zones_list)
axes[1, 0].set_ylabel('Total irrigation (mm)')
axes[1, 0].set_title('Water Savings: Always-On vs Optimised')
axes[1, 0].legend()

# ── Bottom centre: Monte Carlo shortage probability ──
mc_probs = []
for zone in zones_list:
    zp  = zone_params[zone]
    S0  = soil[soil['zone_id']==zone]['soil_moisture_pct'].iloc[0]
    mc  = monte_carlo_simulate(S0, Rain, ET, irr_zero,
                               zp['drainage_coefficient'], zp['field_capacity_pct'],
                               zp['min_moisture_pct'], n_scenarios=500, noise_std=0.3, seed=42)
    mc_probs.append(mc['shortage_prob'] * 100)

bars = axes[1, 1].bar(zones_list, mc_probs, color=[colors[z] for z in zones_list], alpha=0.85)
for bar, val in zip(bars, mc_probs):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}%', ha='center', fontweight='bold')
axes[1, 1].set_ylim(0, 100)
axes[1, 1].set_ylabel('Stress probability (%)')
axes[1, 1].set_title('Monte Carlo: Shortage Risk\n(No Irrigation, 500 Scenarios)')

# ── Bottom right: ET vs Rainfall bar ──
axes[1, 2].bar(range(30), Rain, alpha=0.5, color='steelblue', label='Rainfall')
axes[1, 2].plot(range(30), ET, 'orange', linewidth=2, label='ET')
axes[1, 2].set_xlabel('Day'); axes[1, 2].set_ylabel('mm/day')
axes[1, 2].set_title('Water Supply (Rain) vs Demand (ET)')
axes[1, 2].legend()

plt.suptitle('HydroSense-Kenya - Complete Project Summary', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/level6_final_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Final summary plot saved to outputs/level6_final_summary.png')

## 6. Checklist - Everything Required is Complete

| Requirement | Status | Location |
|-------------|--------|----------|
| Problem statement (500–700 words) | ✅ Done | Level_1_Problem_Framing.ipynb |
| Data dictionary with units | ✅ Done | Level_1_Problem_Framing.ipynb |
| compute_et() and water_balance() functions | ✅ Done | Level_1_Problem_Framing.ipynb |
| Loop vs NumPy timing comparison | ✅ Done | Level_2_Vectorization_and_Error.ipynb |
| Floating point demonstration | ✅ Done | Level_2_Vectorization_and_Error.ipynb |
| Error propagation experiment + plot | ✅ Done | Level_2_Vectorization_and_Error.ipynb |
| Bisection implemented from scratch | ✅ Done | src/numerical_methods.py |
| Newton-Raphson implemented from scratch | ✅ Done | src/numerical_methods.py |
| Secant implemented from scratch | ✅ Done | src/numerical_methods.py |
| Convergence comparison table + plot | ✅ Done | Level_3_Numerical_Methods.ipynb |
| Forward, Backward, Central differences | ✅ Done | src/numerical_methods.py |
| Trapezoidal and Simpson integration | ✅ Done | src/numerical_methods.py |
| Gaussian elimination from scratch | ✅ Done | src/numerical_methods.py |
| 7 data quality issues found and fixed | ✅ Done | Level_4_Data_Analysis_and_Visualization.ipynb |
| Every cleaning decision justified | ✅ Done | Level_4_Data_Analysis_and_Visualization.ipynb |
| 5+ scientific visualisations with interpretation | ✅ Done | Level_4_Data_Analysis_and_Visualization.ipynb |
| Euler method simulation | ✅ Done | src/simulation.py |
| RK4 simulation + Euler comparison | ✅ Done | src/simulation.py + Level_5 |
| Monte Carlo (1000 scenarios) | ✅ Done | src/simulation.py + Level_5 |
| Irrigation optimisation | ✅ Done | src/simulation.py + Level_5 |
| AI use log documented | ✅ Done | AI_USE_LOG.md |
| pytest tests for numerical methods | ✅ Done | tests/test_numerical_methods.py |
| pytest tests for water balance | ✅ Done | tests/test_water_balance.py |
| pytest tests for simulation | ✅ Done | tests/test_simulation.py |
| pytest tests for data cleaning | ✅ Done | tests/test_data_cleaning.py |
| README.md with install instructions | ✅ Done | README.md |
| requirements.txt | ✅ Done | requirements.txt |